In [1]:
video_path = r"C:\Users\Ayush\Desktop\Coding World\Multi_Modal_SnatchDetection\Dataset\Video\Snatch_Shoot\new\Shoot_8.mp4"
TARGET_CLASSES = {
    24: "backpack",
    26: "handbag",
    28: "suitcase",
    67: "cell phone"
}

In [2]:
# ============================================================
# MULTI-MODAL SNATCH DETECTION — CORE PIPELINE (FINAL)
# TABLES 1 → 5 (CROWD-READY, REAL-TIME SAFE)
# ============================================================

from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import math

# ============================================================
# CONFIG
# ============================================================

VIDEO_PATH = video_path

OUTPUT_FRAME_ENTITIES = "frame_entities.csv"

import os
# Automatically extracts "test2" from "C:\...\test2.mp4"
VIDEO_ID = os.path.splitext(os.path.basename(VIDEO_PATH))[0]

# Load YOLOv8
model = YOLO("yolov8n.pt")  # switch to yolov8m.pt if needed

# ============================================================
# CLASS CONFIG
# ============================================================

# Persons & vehicles are ACTORS
PERSON_CLASSES = {0}                 # person
VEHICLE_CLASSES = {1, 3}              # bicycle, motorcycle

# Objects (unstable → centroid tracker only)
TARGET_CLASSES = {
    24: "backpack",
    26: "handbag",
    28: "suitcase",
    67: "cell phone"
}
OBJECT_CLASSES = TARGET_CLASSES

# ============================================================
# HELPERS
# ============================================================

def euclidean(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# ============================================================
# VIDEO META
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

if fps <= 1:
    fps = 30  # fallback

print(f"🎥 FPS detected: {fps}")

# ============================================================
# TABLE 1: frame_entities.csv
# (BoT-SORT + Stable Object Tracking)
# ============================================================

frame_entities = []
frame_id = 0

# --- Object tracker memory ---
object_tracker = {}     # object_id → (cx, cy, last_frame)
next_object_id = 0

MAX_OBJ_DIST = 80       # px
MAX_OBJ_GAP = 5         # frames

# --- Run YOLO + BoT-SORT ---
results_generator = model.track(
    source=VIDEO_PATH,
    tracker="botsort.yaml",     # 🔥 CRITICAL UPGRADE
    persist=True,
    conf=0.35,
    iou=0.5,
    stream=True,
    verbose=False
)

print("🚀 Starting tracking with BoT-SORT...")

for result in results_generator:
    frame_id += 1
    timestamp_sec = frame_id / fps

    if result.boxes is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()

    track_ids = (
        result.boxes.id.cpu().numpy()
        if result.boxes.id is not None
        else [None] * len(boxes)
    )

    for i, box in enumerate(boxes):
        cls_id = int(class_ids[i])
        conf = float(confidences[i])
        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bw = x2 - x1
        bh = y2 - y1

        # ----------------------------------------------------
        # PERSON / VEHICLE (BoT-SORT trusted)
        # ----------------------------------------------------
        if cls_id in PERSON_CLASSES or cls_id in VEHICLE_CLASSES:
            if track_ids[i] is None:
                continue

            entity_type = "person"
            entity_id = int(track_ids[i])

            if cls_id in VEHICLE_CLASSES:
                class_name = "motorcycle"
            else:
                class_name = "person"

        # ----------------------------------------------------
        # OBJECT (Custom centroid tracker ONLY)
        # ----------------------------------------------------
        elif cls_id in OBJECT_CLASSES:
            entity_type = "object"
            class_name = OBJECT_CLASSES[cls_id]

            matched_id = None
            min_dist = float("inf")

            for oid, (ox, oy, last_f) in object_tracker.items():
                if frame_id - last_f > MAX_OBJ_GAP:
                    continue

                dist = euclidean((cx, cy), (ox, oy))
                if dist < MAX_OBJ_DIST and dist < min_dist:
                    min_dist = dist
                    matched_id = oid

            if matched_id is not None:
                entity_id = matched_id
            else:
                entity_id = next_object_id
                next_object_id += 1

            object_tracker[entity_id] = (cx, cy, frame_id)

        else:
            continue

        frame_entities.append({
            "video_id": VIDEO_ID,
            "frame_id": frame_id,
            "timestamp_sec": round(timestamp_sec, 4),
            "entity_type": entity_type,
            "entity_id": entity_id,
            "class_name": class_name,
            "confidence": round(conf, 2),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "bbox_center_x": round(cx, 2),
            "bbox_center_y": round(cy, 2),
            "bbox_width": bw,
            "bbox_height": bh
        })

# Save TABLE 1
df_frame_entities = pd.DataFrame(frame_entities)
df_frame_entities.to_csv(OUTPUT_FRAME_ENTITIES, index=False)

print(f"✅ frame_entities.csv saved ({len(df_frame_entities)} rows)")


🎥 FPS detected: 30.0027275206837
🚀 Starting tracking with BoT-SORT...
✅ frame_entities.csv saved (1310 rows)


In [3]:
# -----------------------------
# VISUALIZATION CELL
# -----------------------------

import cv2
import numpy as np

# Colors
PERSON_COLOR = (0, 255, 0)     # Green
OBJECT_COLOR = (0, 0, 255)     # Red

# Re-open video for clean visualization
cap = cv2.VideoCapture(video_path)

results_generator = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    persist=True,
    conf=0.4,
    stream=True,
    verbose=False
)

frame_id = 0

for result in results_generator:
    frame_id += 1
    frame = result.orig_img.copy()

    if result.boxes is None:
        cv2.imshow("Snatch Detection Debug View", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    track_ids = (
        result.boxes.id.cpu().numpy()
        if result.boxes.id is not None
        else [None] * len(boxes)
    )

    for i, box in enumerate(boxes):
        cls_id = int(class_ids[i])
        conf = float(confidences[i])
        x1, y1, x2, y2 = map(int, box)

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        # ---------------------------
        # PERSON
        # ---------------------------
        if cls_id == 0:
            if track_ids[i] is None:
                continue

            person_id = int(track_ids[i])
            label = f"Person {person_id}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), PERSON_COLOR, 2)
            cv2.putText(frame, label, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, PERSON_COLOR, 2)
            cv2.circle(frame, (cx, cy), 3, PERSON_COLOR, -1)

        # ---------------------------
        # OBJECTS
        # ---------------------------
        elif cls_id in TARGET_CLASSES:
            label = TARGET_CLASSES[cls_id]

            cv2.rectangle(frame, (x1, y1), (x2, y2), OBJECT_COLOR, 2)
            cv2.putText(frame, label, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, OBJECT_COLOR, 2)
            cv2.circle(frame, (cx, cy), 3, OBJECT_COLOR, -1)

    cv2.imshow("Snatch Detection Debug View", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [4]:
df_frame_entities = pd.DataFrame(frame_entities)
# df_frame_entities.to_csv("frame_entities.csv", index=False)
objects_df = df_frame_entities[df_frame_entities['entity_type'] == "object"]
print("All OBJECT detections:")
print(objects_df[['entity_type', 'entity_id', 'class_name']])

All OBJECT detections:
     entity_type  entity_id class_name
3         object          0   backpack
7         object          0   backpack
11        object          0   backpack
15        object          0   backpack
19        object          0   backpack
...          ...        ...        ...
1031      object          2   backpack
1044      object          3    handbag
1047      object          3    handbag
1050      object          3    handbag
1176      object          4    handbag

[230 rows x 3 columns]


In [5]:
# -----------------------------
# TABLE 2: person_motion.csv
# WITH PROPER PERSON MOTION DETECTION
# -----------------------------


import pandas as pd
import numpy as np
import math

# Load raw entities
df_entities = pd.read_csv(r"C:\Users\Ayush\Desktop\Coding World\Multi_Modal_SnatchDetection\frame_entities.csv")

# Keep only persons
df_persons = df_entities[df_entities["entity_type"] == "person"].copy()

# Sort to ensure correct temporal order
df_persons.sort_values(by=["entity_id", "frame_id"], inplace=True)

# --------------------------------------------------------- 
# 🔧 UPGRADE 1: EMA Smoothing (No Lag, Removes Jitter)
# ---------------------------------------------------------
alpha = 0.6  # Responsiveness: 0.5-0.7 ideal for tracking
df_persons['smooth_cx'] = df_persons.groupby('entity_id')['bbox_center_x'] \
    .transform(lambda x: x.ewm(alpha=alpha, adjust=False).mean())
df_persons['smooth_cy'] = df_persons.groupby('entity_id')['bbox_center_y'] \
    .transform(lambda x: x.ewm(alpha=alpha, adjust=False).mean())

# Initialize output and state tracking
person_motion_rows = []
prev_state = {}

# Process each frame chronologically
for _, row in df_persons.iterrows():
    pid = row["entity_id"]
    frame_id = row["frame_id"]
    
    # Use EMA-smoothed coordinates (more responsive than rolling mean)
    cx = row["smooth_cx"]
    cy = row["smooth_cy"]

    if pid not in prev_state:
        # First appearance of person
        vx = vy = speed = accel = direction = 0.0
        prev_state[pid] = {
            "cx": cx, "cy": cy, "speed": 0.0, 
            "direction": 0.0,  # Initialize direction
            "time": row["timestamp_sec"]
        }
    else:
        prev = prev_state[pid]
        
        # Calculate time delta safely
        dt = row["timestamp_sec"] - prev["time"]
        if dt <= 0: 
            dt = 1/30.0  # Assume 30 FPS if timestamp issue
        
        # Velocity from smoothed positions
        vx = (cx - prev["cx"]) / dt
        vy = (cy - prev["cy"]) / dt
        speed = math.sqrt(vx**2 + vy**2)
        
        # Acceleration
        accel = (speed - prev["speed"]) / dt
        
        # 🔧 UPGRADE 2: Stable Direction (Low-Speed Guard)
        if speed < 1e-3:  # Near stationary
            direction = prev["direction"]
        else:
            direction = math.degrees(math.atan2(vy, vx))

        # Update tracking state
        prev_state[pid] = {
            "cx": cx, "cy": cy, "speed": speed, 
            "direction": direction,  # Track stable direction
            "time": row["timestamp_sec"]
        }

    person_motion_rows.append({
        "video_id": VIDEO_ID,
        "frame_id": frame_id,
        "person_id": pid,
        "center_x": round(cx, 2),
        "center_y": round(cy, 2),
        "velocity_x": round(vx, 2),
        "velocity_y": round(vy, 2),
        "speed": round(speed, 2),
        "acceleration": round(accel, 2),
        "direction_angle": round(direction, 2),
        "post_contact_speed": np.nan,    
        "escape_angle_variance": np.nan   
    })

# Save GOLD STANDARD Table 2
df_person_motion = pd.DataFrame(person_motion_rows)
df_person_motion.to_csv("person_motion.csv", index=False)
print("✅ person_motion.csv saved!")
print(df_person_motion.head())
print(f"\n📊 Stats: {len(df_person_motion)} rows, {df_person_motion['person_id'].nunique()} unique persons")


✅ person_motion.csv saved!
  video_id  frame_id  person_id  center_x  center_y  velocity_x  velocity_y  \
0  Shoot_8         1          1    849.00    559.00        0.00        0.00   
1  Shoot_8         2          1    849.60    556.00       17.96      -89.82   
2  Shoot_8         3          1    850.14    553.90       16.22      -63.06   
3  Shoot_8         4          1    850.06    556.06       -2.52       64.86   
4  Shoot_8         5          1    850.32    557.52        7.98       43.83   

   speed  acceleration  direction_angle  post_contact_speed  \
0   0.00          0.00             0.00                 NaN   
1  91.60       2742.49           -78.69                 NaN   
2  65.11       -795.33           -75.58                 NaN   
3  64.91         -6.03            92.23                 NaN   
4  44.55       -609.63            79.69                 NaN   

   escape_angle_variance  
0                    NaN  
1                    NaN  
2                    NaN  
3          

In [6]:
# ============================================================
# TABLE 3: person_pair_interaction.csv
# GEOMETRICALLY ROBUST (FOOTPOINT + PARALLEL LOGIC)
# ============================================================

import pandas as pd
import math

# ------------------------------------------------------------
# 1. Load Data
# ------------------------------------------------------------
df_entities = pd.read_csv("frame_entities.csv")
df_motion   = pd.read_csv("person_motion.csv")

# Keep raw person boxes for footpoint lookup
df_person_boxes = df_entities[df_entities["entity_type"] == "person"] \
    .set_index(["frame_id", "entity_id"])

df_motion.sort_values(by=["frame_id", "person_id"], inplace=True)
grouped = df_motion.groupby("frame_id")

# ------------------------------------------------------------
# 2. Storage
# ------------------------------------------------------------
pair_rows = []
prev_dist = {}

# Thresholds
APPROACH_VEL_THRESH   = -2.0
SEPARATION_VEL_THRESH =  2.0
PARALLEL_THRESH       =  0.8

# ------------------------------------------------------------
# 3. Core Loop
# ------------------------------------------------------------
for frame_id, frame_df in grouped:
    persons = frame_df.to_dict("records")
    n = len(persons)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, n):
            p1, p2 = persons[i], persons[j]
            id1, id2 = int(p1["person_id"]), int(p2["person_id"])
            pair_key = tuple(sorted((id1, id2)))

            # -------------------------------
            # FOOTPOINT DISTANCE (GROUND)
            # -------------------------------
            try:
                b1 = df_person_boxes.loc[(frame_id, id1)]
                b2 = df_person_boxes.loc[(frame_id, id2)]
                f1 = (b1["bbox_center_x"], b1["y2"])
                f2 = (b2["bbox_center_x"], b2["y2"])
            except KeyError:
                f1 = (p1["center_x"], p1["center_y"])
                f2 = (p2["center_x"], p2["center_y"])

            dx = f1[0] - f2[0]
            dy = f1[1] - f2[1]
            dist = math.sqrt(dx*dx + dy*dy) + 1e-6

            d_delta = dist - prev_dist.get(pair_key, dist)
            prev_dist[pair_key] = dist

            # -------------------------------
            # RELATIVE MOTION
            # -------------------------------
            rvx = p1["velocity_x"] - p2["velocity_x"]
            rvy = p1["velocity_y"] - p2["velocity_y"]
            rel_speed = math.sqrt(rvx*rvx + rvy*rvy)

            ux, uy = dx/dist, dy/dist
            approach_vel = (rvx * ux) + (rvy * uy)

            # -------------------------------
            # PARALLEL MOTION SCORE
            # -------------------------------
            if p1["speed"] > 0.5 and p2["speed"] > 0.5:
                dot = (p1["velocity_x"]*p2["velocity_x"] +
                       p1["velocity_y"]*p2["velocity_y"])
                parallel_score = dot / (p1["speed"] * p2["speed"])
            else:
                parallel_score = 0.0

            pair_rows.append({
                "video_id": VIDEO_ID,
                "frame_id": frame_id,
                "person_i_id": id1,
                "person_j_id": id2,
                "distance_ground": round(dist, 2),
                "distance_delta": round(d_delta, 2),
                "relative_speed": round(rel_speed, 2),
                "motion_asymmetry_score": round(abs(p1["speed"] - p2["speed"]), 2),
                "approach_velocity": round(approach_vel, 2),
                "bearing_score": round(parallel_score, 2),
                "approach_flag": int(approach_vel < APPROACH_VEL_THRESH),
                "separation_flag": int(approach_vel > SEPARATION_VEL_THRESH),
                "parallel_flag": int(parallel_score > PARALLEL_THRESH)
            })

# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------
df_pair = pd.DataFrame(pair_rows)
df_pair.to_csv("person_pair_interaction.csv", index=False)

print("✅ person_pair_interaction.csv regenerated (FOOTPOINT + PARALLEL SAFE)")


✅ person_pair_interaction.csv regenerated (FOOTPOINT + PARALLEL SAFE)


In [7]:
# ============================================================
# TABLE 4: hand_interaction.csv
# Grab Signal — Hand to Body Dynamics (FINAL VERSION)
# ============================================================

import pandas as pd
import math

# ------------------------------------------------------------
# 1. Load frame_entities.csv
# ------------------------------------------------------------
df_entities = pd.read_csv("frame_entities.csv")

# Keep only persons
df_persons = df_entities[df_entities["entity_type"] == "person"].copy()
df_persons.sort_values(by=["frame_id", "entity_id"], inplace=True)

# Group by frame
grouped = df_persons.groupby("frame_id")

# ------------------------------------------------------------
# 2. Parameters
# ------------------------------------------------------------
CONTACT_DISTANCE_THRESH = 80     # pixels
MIN_ZONE_RATIO = 0.3             # ignore feet-level contacts
FRAME_GAP_TOLERANCE = 2          # allows jitter / dropped frames

# Memory:
# key = (person_i, person_j, hand_side)
# value = {'count', 'last_frame', 'last_dist'}
contact_memory = {}

hand_rows = []

# ------------------------------------------------------------
# 3. Core Logic
# ------------------------------------------------------------
for frame_id, frame_df in grouped:
    persons = frame_df.to_dict("records")
    n = len(persons)
    if n < 2:
        continue

    for a in range(n):
        for b in range(n):
            if a == b:
                continue

            p_i = persons[a]   # hand owner
            p_j = persons[b]   # body target

            i_id = int(p_i["entity_id"])
            j_id = int(p_j["entity_id"])

            # Victim geometry
            pj_h = p_j["bbox_height"]
            pj_y2 = p_j["y2"]
            body_cx = p_j["bbox_center_x"]
            body_cy = p_j["bbox_center_y"]

            # Approximate left and right hands
            for hand_side, hx_factor in [("left", 0.25), ("right", 0.75)]:
                hand_x = p_i["x1"] + hx_factor * p_i["bbox_width"]
                hand_y = p_i["y1"] + 0.35 * p_i["bbox_height"]

                # Distance hand → body center
                dist = math.sqrt((hand_x - body_cx)**2 + (hand_y - body_cy)**2)
                if dist > CONTACT_DISTANCE_THRESH:
                    continue

                # Contact zone (0=feet, 1=head)
                zone_ratio = (pj_y2 - hand_y) / pj_h
                if zone_ratio < MIN_ZONE_RATIO:
                    continue

                # --- Consecutive contact logic ---
                key = (i_id, j_id, hand_side)

                if key in contact_memory:
                    mem = contact_memory[key]
                    if frame_id - mem["last_frame"] <= FRAME_GAP_TOLERANCE:
                        count = mem["count"] + 1
                        dist_delta = dist - mem["last_dist"]
                    else:
                        count = 1
                        dist_delta = 0.0
                else:
                    count = 1
                    dist_delta = 0.0

                # Update memory
                contact_memory[key] = {
                    "count": count,
                    "last_frame": frame_id,
                    "last_dist": dist
                }

                hand_rows.append({
                    "video_id": VIDEO_ID,
                    "frame_id": frame_id,
                    "person_i_id": i_id,
                    "person_j_id": j_id,
                    "hand_side": hand_side,
                    "hand_center_x": round(hand_x, 2),
                    "hand_center_y": round(hand_y, 2),
                    "distance_hand_to_person_j": round(dist, 2),
                    "distance_hand_delta": round(dist_delta, 2),
                    "contact_duration_frames": count,
                    "contact_zone_ratio": round(zone_ratio, 2)
                })

# ------------------------------------------------------------
# 4. Save TABLE-4
# ------------------------------------------------------------
df_hand_interaction = pd.DataFrame(hand_rows)
df_hand_interaction.to_csv("hand_interaction.csv", index=False)

print("✅ hand_interaction.csv saved (FINAL / SNATCH-READY)")
print(df_hand_interaction.head())
print(f"\n📊 Total hand interactions: {len(df_hand_interaction)}")


✅ hand_interaction.csv saved (FINAL / SNATCH-READY)
  video_id  frame_id  person_i_id  person_j_id hand_side  hand_center_x  \
0  Shoot_8         1            2            3     right         613.00   
1  Shoot_8         2            2            3     right         613.75   
2  Shoot_8         3            2            3     right         613.00   
3  Shoot_8         4            2            3     right         613.00   
4  Shoot_8         5            2            3     right         612.25   

   hand_center_y  distance_hand_to_person_j  distance_hand_delta  \
0         577.55                      78.94                 0.00   
1         578.20                      77.98                -0.95   
2         577.55                      78.94                 0.95   
3         577.55                      78.94                 0.00   
4         577.55                      79.16                 0.23   

   contact_duration_frames  contact_zone_ratio  
0                        1             

In [8]:
# ============================================================
# TABLE 5: object_interaction.csv
# Ownership & Object Loss (FINAL – CLEAN & EVENT-BASED)
# ============================================================

import pandas as pd
import math

# ------------------------------------------------------------
# 1. Load Data
# ------------------------------------------------------------
df_entities = pd.read_csv("frame_entities.csv")
df_motion   = pd.read_csv("person_motion.csv")

# Keep objects only
df_objects = df_entities[df_entities["entity_type"] == "object"].copy()

# Group by frame
objects_by_frame = df_objects.groupby("frame_id")
persons_by_frame = df_motion.groupby("frame_id")

# ------------------------------------------------------------
# 2. Parameters
# ------------------------------------------------------------
MAX_OWNER_DIST = 150      # pixels
DISAPPEAR_GAP  = 3        # frames
MAX_VEL_GAP    = 5        # frames (velocity validity)
EPS = 1e-6

# ------------------------------------------------------------
# 3. Object Memory
# ------------------------------------------------------------
# object_id -> memory
object_memory = {}
object_rows = []

# ------------------------------------------------------------
# 4. Core Loop
# ------------------------------------------------------------
all_frames = sorted(df_entities["frame_id"].unique())

for frame_id in all_frames:

    frame_objects = objects_by_frame.get_group(frame_id) \
        if frame_id in objects_by_frame.groups else pd.DataFrame()

    frame_persons = persons_by_frame.get_group(frame_id) \
        if frame_id in persons_by_frame.groups else pd.DataFrame()

    current_object_ids = set()

    # --------------------------------------------------------
    # 4.1 Process Visible Objects
    # --------------------------------------------------------
    for _, obj in frame_objects.iterrows():
        obj_id = int(obj["entity_id"])
        current_object_ids.add(obj_id)

        cx, cy = obj["bbox_center_x"], obj["bbox_center_y"]
        class_name = obj["class_name"]

        # --- Find Nearest Person (Owner) ---
        owner_id = -1
        owner_dist = float("inf")
        owner_vx = owner_vy = 0.0

        for _, person in frame_persons.iterrows():
            dx = cx - person["center_x"]
            dy = cy - person["center_y"]
            dist = math.sqrt(dx*dx + dy*dy)

            if dist < owner_dist and dist < MAX_OWNER_DIST:
                owner_dist = dist
                owner_id = int(person["person_id"])
                owner_vx = person["velocity_x"]
                owner_vy = person["velocity_y"]

        # --- Object Velocity ---
        if obj_id in object_memory:
            prev = object_memory[obj_id]
            dt = frame_id - prev["last_frame"]

            if 0 < dt <= MAX_VEL_GAP:
                ovx = (cx - prev["last_cx"]) / dt
                ovy = (cy - prev["last_cy"]) / dt
                dist_delta = (
                    owner_dist - prev["last_dist"]
                    if owner_dist != float("inf") and prev["last_dist"] != float("inf")
                    else 0.0
                )
            else:
                ovx = ovy = 0.0
                dist_delta = 0.0
        else:
            ovx = ovy = 0.0
            dist_delta = 0.0

        # --- Velocity Match (Cosine Similarity) ---
        if owner_id != -1:
            dot = ovx * owner_vx + ovy * owner_vy
            mag_o = math.sqrt(ovx**2 + ovy**2)
            mag_p = math.sqrt(owner_vx**2 + owner_vy**2)

            velocity_match = dot / (mag_o * mag_p) if mag_o > EPS and mag_p > EPS else 0.0
        else:
            velocity_match = 0.0

        # --- Update Memory ---
        object_memory[obj_id] = {
            "last_frame": frame_id,
            "last_cx": cx,
            "last_cy": cy,
            "last_person": owner_id,
            "last_dist": owner_dist,
            "class_name": class_name,
            "status": "tracked"
        }

        object_rows.append({
            "video_id": VIDEO_ID,
            "frame_id": frame_id,
            "object_id": obj_id,
            "class_name": class_name,
            "person_id": owner_id,
            "distance_object_to_person": round(owner_dist, 2) if owner_dist != float("inf") else -1,
            "distance_delta": round(dist_delta, 2),
            "present_flag": 1,
            "disappearance_flag": 0,
            "attacker_velocity_match": round(velocity_match, 2)
        })

    # --------------------------------------------------------
    # 4.2 Handle Disappearance (EVENT-BASED)
    # --------------------------------------------------------
    for obj_id, mem in object_memory.items():

        if obj_id in current_object_ids:
            continue

        if mem["status"] == "lost":
            continue

        gap = frame_id - mem["last_frame"]

        if gap > DISAPPEAR_GAP:
            object_rows.append({
                "video_id": VIDEO_ID,
                "frame_id": frame_id,
                "object_id": obj_id,
                "class_name": mem["class_name"],
                "person_id": mem["last_person"],
                "distance_object_to_person": -1,
                "distance_delta": 0.0,
                "present_flag": 0,
                "disappearance_flag": 1,
                "attacker_velocity_match": 0.0
            })

            mem["status"] = "lost"



# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------
df_object_interaction = pd.DataFrame(object_rows)
# In your Table 5 script, right before saving:
if df_object_interaction.empty:
    # Create an empty dataframe with the exact columns expected
    df_object_interaction = pd.DataFrame(columns=[
        "video_id", "frame_id", "object_id", "class_name", "person_id", 
        "distance_object_to_person", "distance_delta", "present_flag", 
        "disappearance_flag", "attacker_velocity_match"
    ])
df_object_interaction.to_csv("object_interaction.csv", index=False)
df_object_interaction.to_csv("object_interaction.csv", index=False)

print("✅ object_interaction.csv saved (FINAL & CLEAN)")
print(df_object_interaction.head())
print(f"\n📊 Total rows: {len(df_object_interaction)}")


✅ object_interaction.csv saved (FINAL & CLEAN)
  video_id  frame_id  object_id class_name  person_id  \
0  Shoot_8         1          0   backpack          1   
1  Shoot_8         2          0   backpack          1   
2  Shoot_8         3          0   backpack          1   
3  Shoot_8         4          0   backpack          1   
4  Shoot_8         5          0   backpack          1   

   distance_object_to_person  distance_delta  present_flag  \
0                      83.00            0.00             1   
1                      80.57           -2.43             1   
2                      78.30           -2.27             1   
3                      80.46            2.16             1   
4                      82.28            1.82             1   

   disappearance_flag  attacker_velocity_match  
0                   0                     0.00  
1                   0                     0.20  
2                   0                    -0.51  
3                   0                    

In [9]:
# ============================================================
# TABLE 6: ML FEATURE AGGREGATION (SLIDING WINDOW) - FINAL SAFE
# ============================================================

import pandas as pd
from pandas.errors import EmptyDataError

print("🚀 Aggregating frame-level data into sliding windows for ML...")

# -----------------------------
# CONFIG
# -----------------------------
VIDEO_ID = "video_1"   # make sure this exists at top also
WINDOW_SIZE_FRAMES = 30
STEP_SIZE_FRAMES = 15

# -----------------------------
# SAFE CSV LOADER
# -----------------------------
def safe_read_csv(path):
    try:
        df = pd.read_csv(path)
        if df.empty:
            print(f"⚠️ {path} is empty")
            return pd.DataFrame()
        print(f"✅ Loaded {path} with columns: {list(df.columns)}")
        return df
    except (FileNotFoundError, EmptyDataError):
        print(f"❌ Missing file: {path}")
        return pd.DataFrame()

# -----------------------------
# LOAD DATA
# -----------------------------
df_motion = safe_read_csv("person_motion.csv")
df_pair   = safe_read_csv("person_pair_interaction.csv")
df_hand   = safe_read_csv("hand_interaction.csv")
df_object = safe_read_csv("object_interaction.csv")

# -----------------------------
# GET MAX FRAME SAFELY
# -----------------------------
max_frame = 0

for df in [df_motion, df_pair, df_hand, df_object]:
    if not df.empty and 'frame_id' in df.columns:
        max_frame = max(max_frame, df['frame_id'].max())

if max_frame == 0:
    raise ValueError("❌ No valid frame_id found in any table")

# -----------------------------
# WINDOW GENERATION
# -----------------------------
window_features = []

for start_frame in range(1, int(max_frame), STEP_SIZE_FRAMES):
    end_frame = start_frame + WINDOW_SIZE_FRAMES
    window_id = f"{VIDEO_ID}_w{start_frame}_{end_frame}"

    # -----------------------------
    # SAFE WINDOW FILTERING
    # -----------------------------
    def filter_df(df):
        if not df.empty and 'frame_id' in df.columns:
            return df[(df['frame_id'] >= start_frame) & (df['frame_id'] <= end_frame)]
        return pd.DataFrame()

    w_motion = filter_df(df_motion)
    w_pair   = filter_df(df_pair)
    w_hand   = filter_df(df_hand)
    w_object = filter_df(df_object)

    if w_motion.empty and w_object.empty:
        continue

    # -----------------------------
    # FEATURE EXTRACTION (SAFE)
    # -----------------------------
    features = {
        "video_id": VIDEO_ID,
        "window_id": window_id,
        "start_frame": start_frame,
        "end_frame": end_frame,

        # Motion
        "avg_speed": round(w_motion['speed'].mean(), 2) if not w_motion.empty else 0.0,
        "max_speed": min(round(w_motion['speed'].max(), 2), 300) if not w_motion.empty else 0.0,
        "max_acceleration": min(round(w_motion['acceleration'].max(), 2), 5000) if not w_motion.empty else 0.0,

        # Pair Interaction
        "min_distance_ground": round(w_pair['distance_ground'].min(), 2) if not w_pair.empty else 999.0,
        "avg_approach_velocity": round(w_pair['approach_velocity'].mean(), 2) if not w_pair.empty else 0.0,
        "max_relative_speed": round(w_pair['relative_speed'].max(), 2) if not w_pair.empty else 0.0,

        # Hand Interaction
        "max_contact_duration": int(w_hand['contact_duration_frames'].max()) if not w_hand.empty else 0,
        "contact_events_count": int(w_hand['person_i_id'].nunique()) if not w_hand.empty else 0,

        # Object Interaction
        "object_lost_flag": 1 if (not w_object.empty and 'disappearance_flag' in w_object.columns and (w_object['disappearance_flag'] == 1).any()) else 0,
        "max_velocity_match": round(w_object['attacker_velocity_match'].max(), 2) if (not w_object.empty and 'attacker_velocity_match' in w_object.columns) else 0.0,

        # Label (manual later)
        "label": 0
    }

    window_features.append(features)

# -----------------------------
# SAVE OUTPUT
# -----------------------------
df_features = pd.DataFrame(window_features)

output_filename = f"{VIDEO_ID}_features.csv"
df_features.to_csv(output_filename, index=False)

print(f"\n✅ {output_filename} saved successfully")
print(f"📊 Total windows generated: {len(df_features)}")
print(df_features.head())

🚀 Aggregating frame-level data into sliding windows for ML...
✅ Loaded person_motion.csv with columns: ['video_id', 'frame_id', 'person_id', 'center_x', 'center_y', 'velocity_x', 'velocity_y', 'speed', 'acceleration', 'direction_angle', 'post_contact_speed', 'escape_angle_variance']
✅ Loaded person_pair_interaction.csv with columns: ['video_id', 'frame_id', 'person_i_id', 'person_j_id', 'distance_ground', 'distance_delta', 'relative_speed', 'motion_asymmetry_score', 'approach_velocity', 'bearing_score', 'approach_flag', 'separation_flag', 'parallel_flag']
✅ Loaded hand_interaction.csv with columns: ['video_id', 'frame_id', 'person_i_id', 'person_j_id', 'hand_side', 'hand_center_x', 'hand_center_y', 'distance_hand_to_person_j', 'distance_hand_delta', 'contact_duration_frames', 'contact_zone_ratio']
✅ Loaded object_interaction.csv with columns: ['video_id', 'frame_id', 'object_id', 'class_name', 'person_id', 'distance_object_to_person', 'distance_delta', 'present_flag', 'disappearance_fl

In [10]:









# Old Table 6:

# import pandas as pd

# import pandas as pd
# from pandas.errors import EmptyDataError

# # ------------------------------------------------------------
# # Safe CSV Loader
# # ------------------------------------------------------------
# def safe_read_csv(path, required_columns):
#     try:
#         df = pd.read_csv(path)
#         if df.empty:
#             raise EmptyDataError
#         return df
#     except (FileNotFoundError, EmptyDataError):
#         print(f"⚠️ Warning: {path} is empty or missing. Using empty fallback.")
#         return pd.DataFrame(columns=required_columns)

# # ------------------------------------------------------------
# # Load Tables (SAFE)
# # ------------------------------------------------------------
# df_pair = safe_read_csv(
#     "person_pair_interaction.csv",
#     [
#         "frame_id","person_i_id","person_j_id","distance_ground",
#         "distance_delta","relative_speed","motion_asymmetry_score",
#         "approach_velocity","bearing_score",
#         "approach_flag","separation_flag","parallel_flag"
#     ]
# )

# df_hand = safe_read_csv(
#     "hand_interaction.csv",
#     [
#         "frame_id","person_i_id","person_j_id","hand_side",
#         "hand_center_x","hand_center_y",
#         "distance_hand_to_person_j","distance_hand_delta",
#         "contact_duration_frames","contact_zone_ratio"
#     ]
# )

# df_object = safe_read_csv(
#     "object_interaction.csv",
#     [
#         "frame_id","object_id","class_name","person_id",
#         "distance_object_to_person","distance_delta",
#         "present_flag","disappearance_flag",
#         "attacker_velocity_match"
#     ]
# )

# # ------------------------------------------------------------
# # Parameters
# # ------------------------------------------------------------
# WINDOW_RADIUS   = 15
# MIN_CONFIDENCE  = 0.50

# # Weights
# W_APPROACH   = 0.13
# W_SEPARATION = 0.23
# W_HAND       = 0.24
# W_OBJECT     = 0.18
# W_VEL_MATCH  = 0.22

# # ------------------------------------------------------------
# # Anchors
# # ------------------------------------------------------------
# anchors = []

# anchors += [{"frame": int(r.frame_id), "src": "object"}
#             for _, r in df_object[df_object.disappearance_flag == 1].iterrows()]

# anchors += [{"frame": int(r.frame_id), "src": "hand"}
#             for _, r in df_hand[df_hand.contact_duration_frames <= 8].iterrows()]

# anchors = sorted(anchors, key=lambda x: x["frame"])

# # ------------------------------------------------------------
# # Evaluate Windows
# # ------------------------------------------------------------
# events = []
# used = set()
# eid = 0

# for anc in anchors:
#     f = anc["frame"]
#     if any(abs(f - u) <= WINDOW_RADIUS for u in used):
#         continue
#     used.add(f)

#     s, e = f - WINDOW_RADIUS, f + WINDOW_RADIUS
#     wp = df_pair[(df_pair.frame_id >= s) & (df_pair.frame_id <= e)]
#     wh = df_hand[(df_hand.frame_id >= s) & (df_hand.frame_id <= e)]
#     wo = df_object[(df_object.frame_id >= s) & (df_object.frame_id <= e)]

#     # ---------------- NEGATIVE GATES ----------------
#     if not wp.empty and wp.parallel_flag.mean() > 0.6:
#         continue  # Side-by-side walking

#     if not wh.empty and wh.contact_duration_frames.max() > 30:
#         continue  # Handshake / hug

#     # ---------------- POSITIVE EVIDENCE ----------------
#     confidence = 0.0
#     evidence = []

#     if (wp.approach_flag == 1).any():
#         confidence += W_APPROACH
#         evidence.append("approach")

#     if (wp.separation_flag == 1).any() and wp.relative_speed.max() > 120:
#         confidence += W_SEPARATION
#         evidence.append("escape")

#     if not wh.empty and wh.contact_duration_frames.min() <= 8:
#         confidence += W_HAND
#         evidence.append("grab")

#     if wo.disappearance_flag.any():
#         confidence += W_OBJECT
#         evidence.append("object_lost")

#     if wo.attacker_velocity_match.max() > 0.5:
#         confidence += W_VEL_MATCH
#         evidence.append("velocity_match")

#     confidence = min(confidence, 1.0)

#     if confidence >= MIN_CONFIDENCE:
#         events.append({
#             "event_id": eid,
#             "start_frame": s,
#             "end_frame": e,
#             "confidence_score": round(confidence, 2),
#             "evidence_flags": " | ".join(evidence)
#         })
#         eid += 1

# # ------------------------------------------------------------
# # Save
# # ------------------------------------------------------------
# df_events = pd.DataFrame(events)
# df_events.to_csv("event_window.csv", index=False)